# 07 - Logistic regression baseline (RQ1)

Rung 3 of the baseline ladder: TF-IDF features + L2 logistic regression, one
binary model per construct (1A, 1B, 2). A discriminative model that estimates
the decision boundary directly, rather than the class-conditional word
frequencies of Naïve Bayes. Features are fit on `train_fit` and scored on
`validation`; the held-out test set is scored once, later, in the final build.
Class weights are balanced to counteract construct imbalance, and stop-word
removal is **disabled** in the reported model (pronouns / in-group markers are
diagnostic); a stop-word ablation is included at the end for the appendix.

In [ ]:
# Imports and load 

import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Repo root, whether run from notebooks/ or the root (no hard-coded local paths).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
proc = PROJECT_ROOT / "data" / "processed"

# Fit on train_fit (1215), score on validation (405). Test stays held out.
train = pd.read_csv(proc / "train_fit.csv")
val   = pd.read_csv(proc / "validation.csv")
label_cols = ["label_1A", "label_1B", "label_2"]
for c in label_cols:
    train[c] = train[c].astype(int)
    val[c]   = val[c].astype(int)

In [ ]:
# Reported model (no-stop words)

# ---- Features: TF-IDF unigrams+bigrams, fit on TRAIN ONLY ----
# stop_words=None: keep pronouns / in-group markers, which are diagnostic here.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=3,             # ignore terms in fewer than 3 comments
    max_df=0.9,           # ignore terms in >90% of comments
    stop_words=None,
)
X_train = vectorizer.fit_transform(train["text_clean"].fillna(""))  # fit on train
X_val   = vectorizer.transform(val["text_clean"].fillna(""))        # transform only on val
print("train:", X_train.shape, "| val:", X_val.shape)

# ---- One balanced binary model per construct ----
rows = []
for label in label_cols:
    model = LogisticRegression(
        class_weight="balanced",   # up-weight the rare positive class
        max_iter=1000,             # enough iterations to converge
    )
    model.fit(X_train, train[label])
    pred = model.predict(X_val)
    rows.append({
        "construct": label.replace("label_", ""),
        "precision": round(precision_score(val[label], pred, zero_division=0), 3),
        "recall":    round(recall_score(val[label], pred, zero_division=0), 3),
        "f1":        round(f1_score(val[label], pred, zero_division=0), 3),
        "support":   int(val[label].sum()),
    })
lr_results = pd.DataFrame(rows)
print("\n", lr_results.to_string(index=False))
lr_results.to_csv(proc / "lr_validation_metrics.csv", index=False)   # reported figures

### Stop-word ablation (appendix)

The same TF-IDF + logistic pipeline with English stop-words removed, for
comparison only. This is **not** the reported configuration; it is kept to show
the effect of discarding pronouns and function words. Saved to a separate file
so it does not overwrite the reported metrics above.

In [ ]:
# Identical pipeline, stop_words="english" — appendix ablation only.
vec_sw = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=3, max_df=0.9,
                         stop_words="english")
Xtr_sw = vec_sw.fit_transform(train["text_clean"].fillna(""))
Xva_sw = vec_sw.transform(val["text_clean"].fillna(""))

rows = []
for label in label_cols:
    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(Xtr_sw, train[label])
    pred = model.predict(Xva_sw)
    rows.append({
        "construct": label.replace("label_", ""),
        "precision": round(precision_score(val[label], pred, zero_division=0), 3),
        "recall":    round(recall_score(val[label], pred, zero_division=0), 3),
        "f1":        round(f1_score(val[label], pred, zero_division=0), 3),
        "support":   int(val[label].sum()),
    })
lr_results_sw = pd.DataFrame(rows)
print(lr_results_sw.to_string(index=False))
lr_results_sw.to_csv(proc / "lr_validation_metrics_stopwords.csv", index=False)   # ablation